In this competition we will be using data generated by a deep learning model trained on the [California housing dataset](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_california_housing.html). We can expect the relationships between variables to be similar as in the original dataset, but not exactly the same.

We will be predicting the the median house value for California districts, expressed in hundreds of thousands of dollars ($100,000). The independent variables at our disposal are:
* `MedInc` - median income in block group
* `HouseAge` - median house age in block group
* `AveRooms` - average number of rooms per household
* `AveBedrms` - average number of bedrooms per household
* `Population` - block group population
* `AveOccup` - average number of household members
* `Latitude` - block group latitude
* `Longitude` - block group longitude

The evaluation metric is going the be the standard Root Mean Squared Error (RMSE) and the useful thing to keep in mind about this metric, as it involves a squared term, is that outliers, or predictions that err a lot, are disproportionately penalized!

Let's look at the data.

In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np

In [ ]:
DATAPATH = Path('../input/playground-series-s3e1')
N_ESTIMATORS = 100_000

train = pd.read_csv(DATAPATH/'train.csv')
test = pd.read_csv(DATAPATH/'test.csv')
sample_sub = pd.read_csv(DATAPATH/'sample_submission.csv')

All columns seem to be numeric, no sign of categorical columns.

In [ ]:
train.head()

The id column seems to be an artifact of preprocessing, there are no duplicate entries for it.

In [ ]:
train.id.value_counts().max()

The big question -- do we have any NAs?

And the answer is no!

In [ ]:
train.isna().sum()

A relatively sizable train and test set. Should make for a fun competition!

In [ ]:
train.shape[0], test.shape[0]

Ok, so seems like we have an interesting competition on our hands.

We could look at the distribution of variables and all the good stuff, but let instead have our model handle this! At this stage, we probably can get to better insights faster via querying the model than performing the analysis manually from scratch.

# Training our first model (with basic ensembling!)

Let's begin by splitting our data into a train and validation set.

In [ ]:
from lightgbm.sklearn import LGBMRegressor
import lightgbm as lgbm
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold

The variable that we will be predicting is the `MedHouseVal`. We will use the rest of the columns (minus the id column) for training.

In [ ]:
features = ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
target = 'MedHouseVal'

A very interesting finding that people are reporting is that adding original data (the data on which the Deep Learning model that was used to generate the data for this competition was trained) helps.

Let's download the data and add it to our training data.

In [ ]:
from sklearn.datasets import fetch_california_housing

original_data = fetch_california_housing()
train = train.drop(columns='id')
original_data = pd.DataFrame(data=np.hstack([original_data['data'], original_data['target'].reshape(-1, 1)]), columns=train.columns)

train = pd.concat([train, original_data]).reset_index(drop=True)

The parameters I am using to train the lightgbm model come from this [awesome notebook](https://www.kaggle.com/code/soupmonster/simple-lightgbm-baseline) by soupmonster.

In [ ]:
clfs = []
rmses = []

params= {
 'lambda_l1': 1.945,
 'num_leaves': 87,
 'feature_fraction': 0.79,
 'bagging_fraction': 0.93,
 'bagging_freq': 4,
 'min_data_in_leaf': 103,
 'max_depth': 17,
}

kf = KFold(n_splits=10, random_state=0, shuffle=True)
for train_index, val_index in kf.split(train):
    X_train, X_val = train[features].loc[train_index], train[features].loc[val_index]
    y_train, y_val = train[target][train_index], train[target][val_index]
    
    clf = LGBMRegressor(learning_rate=0.02, n_estimators=N_ESTIMATORS, metric='rmse', **params)
    clf.fit(X_train.values, y_train, eval_set=[(X_val, y_val)], callbacks=[lgbm.early_stopping(85, verbose=True)])
    preds = clf.predict(X_val.values)
    
    clfs.append(clf)
    rmses.append(mean_squared_error(y_val, preds, squared=False))
print(f'mean RMSE across all folds: {np.mean(rmses)}')

Let us now look at the variables that are important according to our model.

In [ ]:
for i in clf.feature_importances_.argsort()[::-1]:
    print(features[i], clf.feature_importances_[i]/clf.feature_importances_.sum())

`Longitude`, `Latitude`, `MedInc` (medium income in block group) seem to be among the most important ones and `AveOccup` (average number of household members). That is a good staritng point, that is probably where we should direct our attention initially.

But bear in mind that tree based models are not great with identifying interactions! Maybe interactions between variables is where a wealth of signal resides? With just 8 variables exploring interactions is tractable and can lead to a significant boost in performance!

But you know what could be fun? Let's test our simple ensembling idea by adding another model to the mix.

Let's see how far we can fare if we mix the lightgbm model with a catboost model.

In [ ]:
from catboost import CatBoostRegressor

rmses = []
kf = KFold(n_splits=10, random_state=1, shuffle=True)
for train_index, val_index in kf.split(train):
    X_train, X_val = train[features].loc[train_index], train[features].loc[val_index]
    y_train, y_val = train[target][train_index], train[target][val_index]

    clf = CatBoostRegressor(iterations=N_ESTIMATORS, loss_function='RMSE')
    clf.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=1000, verbose=False)
    
    
    preds = clf.predict(X_val.values)
    
    clfs.append(clf)
    rmses.append(mean_squared_error(y_val, preds, squared=False))
print(f'mean RMSE across all folds: {np.mean(rmses)}')

Seems catboost can achieve amazing performance without hyperparameter tuning but it comes at a price of long computation!

# Making a submission

Lets being by predicting on the test set using each of our trained classifiers.

In [ ]:
test_preds = []

for clf in clfs:
    preds = clf.predict(test[features].values)
    test_preds.append(preds)

We can now take the mean of our predictions

In [ ]:
test_preds = np.stack(test_preds).mean(0)
test_preds

Looking good! Let's generate a submission.

In [ ]:
submission = pd.DataFrame(data={'id': test.id, 'MedHouseVal': test_preds})
submission.head()

In [ ]:
submission.to_csv('submission.csv', index=False)

This is shaping up to be a very excting challenge! 🥳 

**If you found this notebook useful, please upvote! 🙏 Thank you!**

All the best in the competition!